# Hafta 3: Google Trends Analizi

Bu defterde **pytrends** kütüphanesi ile Google Trends verilerini çekip analiz edeceğiz.

## İçindekiler
1. Kütüphane kurulumu ve ayarlar
2. Anahtar kelime araması
3. Zaman içinde ilgi grafiği
4. Bölgelere göre ilgi
5. İlgili sorgular
6. Karşılaştırma grafiği

## 1. Kütüphane Kurulumu ve Ayarlar

### pytrends kütüphanesini yükle

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# pytrends kütüphanesini yükle
!pip install pytrends

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `matplotlib` | Grafik ve görselleştirme |
| `numpy` | Sayısal hesaplamalar ve dizi işlemleri |
| `pandas` | Veri çerçeveleri (DataFrame) ile veri analizi |
| `pytrends` | Google Trends API erişimi |
| `seaborn` | İstatistiksel görselleştirme (Matplotlib üzerine kurulu) |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pytrends.request import TrendReq

%matplotlib inline
sns.set_style("whitegrid")

# pytrends bağlantısını oluştur
pytrend = TrendReq(hl='tr-TR', tz=180)  # Türkçe, UTC+3 (Türkiye)

print("Kütüphaneler ve Google Trends bağlantısı hazır!")

## 2. Anahtar Kelime Araması

Araştıracağımız anahtar kelimeler:
- **Yapay Zeka**
- **Metaverse**
- **ChatGPT**

In [ ]:
# Anahtar kelimeleri tanımla
anahtar_kelimeler = ["Yapay Zeka", "Metaverse", "ChatGPT"]

# Google Trends'ten veri çek
# timeframe: 'today 12-m' = son 12 ay, 'today 5-y' = son 5 yıl
pytrend.build_payload(
    kw_list=anahtar_kelimeler,
    cat=0,
    timeframe='today 12-m',  # Son 12 ay
    geo='TR',  # Türkiye
    gprop=''  # Web araması
)

print(f"Aranan kelimeler: {anahtar_kelimeler}")
print("Bölge: Türkiye")
print("Zaman aralığı: Son 12 ay")

## 3. Zaman İçinde İlgi Grafiği (Interest Over Time)

In [ ]:
# Zaman içinde ilgi verisi
ilgi_zamana = pytrend.interest_over_time()

print(f"Veri boyutu: {ilgi_zamana.shape}")
print(f"\nİlk 5 satır:")
ilgi_zamana.head()

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Zaman içinde ilgi çizgi grafiği
plt.figure(figsize=(14, 7))

renkler = {'Yapay Zeka': '#E53935', 'Metaverse': '#1E88E5', 'ChatGPT': '#43A047'}

for kelime in anahtar_kelimeler:
    if kelime in ilgi_zamana.columns:
        plt.plot(ilgi_zamana.index, ilgi_zamana[kelime],
                 linewidth=2.5, label=kelime, color=renkler[kelime], marker='o', markersize=3)

plt.title('Google Trends: Zaman İçinde Arama İlgisi (Türkiye)', fontsize=16, fontweight='bold')
plt.xlabel('Tarih', fontsize=12)
plt.ylabel('Arama İlgisi (0-100)', fontsize=12)
plt.legend(fontsize=13, loc='upper left')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.ylim(0, 105)
plt.tight_layout()
plt.show()

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Alan grafiği olarak da gösterelim
plt.figure(figsize=(14, 7))

for kelime in anahtar_kelimeler:
    if kelime in ilgi_zamana.columns:
        plt.fill_between(ilgi_zamana.index, ilgi_zamana[kelime], alpha=0.3, color=renkler[kelime])
        plt.plot(ilgi_zamana.index, ilgi_zamana[kelime],
                 linewidth=2, label=kelime, color=renkler[kelime])

plt.title('Google Trends: Arama İlgisi (Alan Grafiği)', fontsize=16, fontweight='bold')
plt.xlabel('Tarih', fontsize=12)
plt.ylabel('Arama İlgisi (0-100)', fontsize=12)
plt.legend(fontsize=13)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Bölgelere Göre İlgi (Interest by Region)

### Bölgelere göre ilgi verisi

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Bölgelere göre ilgi verisi
ilgi_bolge = pytrend.interest_by_region(resolution='COUNTRY', inc_low_vol=True, inc_geo_code=False)

# Sıfır olmayan değerleri filtrele
ilgi_bolge = ilgi_bolge[ilgi_bolge.sum(axis=1) > 0]

print(f"Toplam bölge sayısı: {len(ilgi_bolge)}")
print(f"\nEn yüksek ilgi gösteren ilk 10 ülke:")
ilgi_bolge.head(10)

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Her anahtar kelime için en ilgili 15 ülke
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for i, kelime in enumerate(anahtar_kelimeler):
    if kelime in ilgi_bolge.columns:
        en_ilgili = ilgi_bolge[kelime].nlargest(15).sort_values()
        en_ilgili.plot(kind='barh', ax=axes[i], color=renkler[kelime], edgecolor='black', linewidth=0.5)
        axes[i].set_title(f'"{kelime}" - En İlgili 15 Ülke', fontsize=13, fontweight='bold')
        axes[i].set_xlabel('İlgi Skoru', fontsize=11)
        axes[i].set_ylabel('')

plt.suptitle('Google Trends: Bölgelere Göre Arama İlgisi', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Türkiye İlleri Bazında Analiz

Sadece Türkiye içindeki bölgesel ilgiyi de inceleyebiliriz.

In [ ]:
# Türkiye bazında bölgesel ilgi
pytrend.build_payload(
    kw_list=['Yapay Zeka'],
    cat=0,
    timeframe='today 12-m',
    geo='TR',
    gprop=''
)

ilgi_turkiye = pytrend.interest_by_region(resolution='REGION', inc_low_vol=True, inc_geo_code=False)
ilgi_turkiye = ilgi_turkiye[ilgi_turkiye['Yapay Zeka'] > 0].sort_values('Yapay Zeka', ascending=False)

print(f"Toplam il/bölge sayısı: {len(ilgi_turkiye)}")
print("\nEn çok aranan ilk 10 il:")
ilgi_turkiye.head(10)

### Görselleştirme

Aşağıdaki grafikle veriyi görsel olarak inceliyoruz. Görselleştirme, sayısal analizlerin ötesinde örüntüleri ve anormallikleri fark etmemizi sağlar.

In [ ]:
# Türkiye'de "Yapay Zeka" araması - İlk 20 il
en_ilgili_iller = ilgi_turkiye.head(20).sort_values('Yapay Zeka')

plt.figure(figsize=(12, 8))
plt.barh(en_ilgili_iller.index, en_ilgili_iller['Yapay Zeka'],
         color=plt.cm.Reds(np.linspace(0.3, 0.9, len(en_ilgili_iller))),
         edgecolor='black', linewidth=0.5)
plt.title('Türkiye\'de "Yapay Zeka" Araması - En İlgili 20 İl', fontsize=16, fontweight='bold')
plt.xlabel('Arama İlgisi Skoru', fontsize=12)
plt.ylabel('')
plt.tight_layout()
plt.show()

## 5. İlgili Sorgular (Related Queries)

### Ana anahtar kelimeleri tekrar yükle

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Ana anahtar kelimeleri tekrar yükle
pytrend.build_payload(
    kw_list=anahtar_kelimeler,
    cat=0,
    timeframe='today 12-m',
    geo='TR',
    gprop=''
)

# İlgili sorgular
ilgili_sorgular = pytrend.related_queries()

for kelime in anahtar_kelimeler:
    print(f"\n{'='*60}")
    print(f'  "{kelime}" İçin İlgili Sorgular')
    print(f"{'='*60}")
    
    if kelime in ilgili_sorgular:
        # En çok arananlar (Top)
        top = ilgili_sorgular[kelime].get('top')
        if top is not None and not top.empty:
            print(f"\n📊 En Çok Aranan Sorgular:")
            print(top.head(10).to_string(index=False))
        
        # Yükselen sorgular (Rising)
        rising = ilgili_sorgular[kelime].get('rising')
        if rising is not None and not rising.empty:
            print(f"\n🚀 Yükselen Sorgular:")
            print(rising.head(10).to_string(index=False))

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# İlgili sorguları görselleştir
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for i, kelime in enumerate(anahtar_kelimeler):
    if kelime in ilgili_sorgular:
        top = ilgili_sorgular[kelime].get('top')
        if top is not None and not top.empty:
            top10 = top.head(10).sort_values('value')
            axes[i].barh(top10['query'], top10['value'], color=renkler[kelime],
                         edgecolor='black', linewidth=0.5)
            axes[i].set_title(f'"{kelime}"\nİlgili Sorgular', fontsize=13, fontweight='bold')
            axes[i].set_xlabel('İlgi Skoru', fontsize=11)

plt.suptitle('Google Trends: Anahtar Kelimelere İlgili En Popüler Sorgular',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Karşılaştırma Grafiği

Üç anahtar kelimeyi çeşitli açılardan karşılaştıralım.

In [ ]:
# Ortalama ilgi karşılaştırması
pytrend.build_payload(
    kw_list=anahtar_kelimeler,
    cat=0,
    timeframe='today 12-m',
    geo='TR',
    gprop=''
)

ilgi_zamana = pytrend.interest_over_time()

# Ortalama, maksimum, minimum hesapla
istatistikler = pd.DataFrame({
    'Ortalama': ilgi_zamana[anahtar_kelimeler].mean(),
    'Maksimum': ilgi_zamana[anahtar_kelimeler].max(),
    'Minimum': ilgi_zamana[anahtar_kelimeler].min(),
    'Standart Sapma': ilgi_zamana[anahtar_kelimeler].std()
})

print("Anahtar Kelime İstatistikleri:")
print("=" * 60)
print(istatistikler.round(2))

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Karşılaştırma çubuk grafiği
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Ortalama ilgi
ort = ilgi_zamana[anahtar_kelimeler].mean().sort_values(ascending=True)
ort.plot(kind='barh', ax=axes[0, 0], color=[renkler[k] for k in ort.index], edgecolor='black')
axes[0, 0].set_title('Ortalama Arama İlgisi', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('İlgi Skoru')

# 2. Kutu grafiği (dağılım)
ilgi_erime = ilgi_zamana[anahtar_kelimeler].melt(var_name='Kelime', value_name='İlgi')
sns.boxplot(data=ilgi_erime, x='Kelime', y='İlgi',
            palette=renkler, ax=axes[0, 1])
axes[0, 1].set_title('İlgi Dağılımı', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('İlgi Skoru')

# 3. Aylık ortalama çizgi grafiği
aylik = ilgi_zamana[anahtar_kelimeler].resample('M').mean()
for kelime in anahtar_kelimeler:
    if kelime in aylik.columns:
        axes[1, 0].plot(aylik.index, aylik[kelime], marker='o',
                        linewidth=2.5, label=kelime, color=renkler[kelime], markersize=6)
axes[1, 0].set_title('Aylık Ortalama İlgi Trendi', fontsize=13, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].set_ylabel('İlgi Skoru')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# 4. Korelasyon ısı haritası
korelasyon = ilgi_zamana[anahtar_kelimeler].corr()
sns.heatmap(korelasyon, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', linewidths=2, ax=axes[1, 1], square=True,
            annot_kws={'fontsize': 14, 'fontweight': 'bold'})
axes[1, 1].set_title('Anahtar Kelime Korelasyonu', fontsize=13, fontweight='bold')

fig.suptitle('Google Trends Karşılaştırma Paneli', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Temel İstatistikler

Verinin genel yapısını inceliyoruz: sütun tipleri, eksik değerler, temel istatistikler (ortalama, medyan, min, max). Bu bilgiler veri temizleme ve ön işleme adımlarını planlamak için gereklidir.

In [ ]:
# Bonus: Daha uzun dönem analizi (5 yıl)
pytrend.build_payload(
    kw_list=anahtar_kelimeler,
    cat=0,
    timeframe='today 5-y',  # Son 5 yıl
    geo='TR',
    gprop=''
)

ilgi_5yil = pytrend.interest_over_time()

plt.figure(figsize=(16, 7))
for kelime in anahtar_kelimeler:
    if kelime in ilgi_5yil.columns:
        plt.plot(ilgi_5yil.index, ilgi_5yil[kelime],
                 linewidth=2, label=kelime, color=renkler[kelime])
        plt.fill_between(ilgi_5yil.index, ilgi_5yil[kelime], alpha=0.1, color=renkler[kelime])

plt.title('Google Trends: Son 5 Yıllık Arama İlgisi (Türkiye)', fontsize=16, fontweight='bold')
plt.xlabel('Tarih', fontsize=12)
plt.ylabel('Arama İlgisi (0-100)', fontsize=12)
plt.legend(fontsize=13)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n5 Yıllık Dönem İstatistikleri:")
print(ilgi_5yil[anahtar_kelimeler].describe().round(1))

---

## Özet

Bu defterde öğrendiklerimiz:

| Özellik | Fonksiyon | Açıklama |
|---------|-----------|----------|
| Bağlantı | `TrendReq(hl, tz)` | Google Trends API bağlantısı |
| Veri yükleme | `build_payload()` | Anahtar kelime ve parametreleri ayarla |
| Zaman ilgisi | `interest_over_time()` | Zamana göre arama ilgisi |
| Bölge ilgisi | `interest_by_region()` | Bölgelere göre arama ilgisi |
| İlgili sorgular | `related_queries()` | En çok aranan ve yükselen ilgili sorgular |

### Önemli Parametreler

| Parametre | Değer | Açıklama |
|-----------|-------|----------|
| `geo` | `'TR'` | Türkiye |
| `timeframe` | `'today 12-m'` | Son 12 ay |
| `timeframe` | `'today 5-y'` | Son 5 yıl |
| `timeframe` | `'2023-01-01 2024-01-01'` | Belirli tarih aralığı |
| `resolution` | `'REGION'` | Alt bölge düzeyinde analiz |

### İpuçları
- Google Trends verileri **göreceli** değerlerdir (0-100 arası). Mutlak arama sayısı değildir.
- Çok sık sorgu atarsanız Google geçici olarak erişimi kısıtlayabilir.
- Farklı zaman aralıkları farklı sonuçlar verebilir; karşılaştırmalarda aynı aralığı kullanın.